# Electrode Array

In [6]:
from customComponents import ElectrodeArray

# configure stim packet
sd1 = {}
sd1["elecCath"] = [1,12]
sd1["elecAno"] = [3,4]
sd1["Gnd"] = [6,7]
sd1["Ref"] = [13,14]

EA = ElectrodeArray(ElecLoc="Caudal")
EA.update(sd1)
EA.fig.show()

In [7]:
from customComponents import ElectrodeArray

# configure stim packet
sd1 = {}
sd1["elecCath"] = [129,60]
sd1["elecAno"] = [133,145]
sd1["Gnd"] = [141]
sd1["Ref"] = [144]

EA = ElectrodeArray(ElecLoc="Rostral")
EA.update(sd1)
EA.fig.show()

# Stim Group

In [3]:
import pandas as pd
import numpy as np
import dash
import plotly.express as px
import plotly.graph_objects as go

class StimGroup():
    def __init__(self):
        self.df = {}
        self.fig = []
        self.build()

    def build(self):
        fig = go.Figure()
        fig.update_layout({
            'template': 'simple_white',
            'width': 400,
            'height': 300,
            'showlegend': False
        })
        fig.update_xaxes(mirror=True, showticklabels=False, ticks='')
        fig.update_yaxes(mirror=True, showticklabels=False, ticks='')

        # update self 
        self.fig = fig 
        
    def update(self,sgdict):
        """
        update the state the Stim Group
        inputs: 
            a dictionary with the following fields:
                eleCath - [list of cathodes in group]
                elecAno - [list of anodes in group]
                amp   - float
                freq  - float
                phase - float
        """
        self.fig.add_trace(
            go.Scatter(
                x = df['x'].to_list(), 
                y = df['y'].to_list(), 
                text = df['index'].to_list(), 
                textfont=dict(
                    family="sans serif",
                    size=18,
                    color="White"
                ),
                marker_color=df['color'],
                mode='markers+text',
                marker={'size': 40}))
    # scatter plot dataframe 


##
## strategy: 
## figure out how many total rows we need based on total number of pins in SG
## build out the meshgrid
## apply 2 additional rows at the bottom for annotation. 
## annotations on the sides
## let's do it!
#  
    # def update(self,Pins): 
    #     """
    #     update the state the electrode array figure 
    #     inputs: 
    #         a dictionary with the following fields:
    #             eleCath 
    #             elecAno
    #             Gnd
    #             Ref
    #     """
    #     EleGroupsNames = ['elecCath','elecAno','Gnd','Ref']
    #     EleGroupColors = {'elecCath':'red','elecAno':'blue','Gnd':'green','Ref':'purple'}
    #     self.df['color'] = 'gray' # blank the electrodes
        
    #     # update df with proper colors
    #     for groupName in Pins: 
    #         if groupName in EleGroupsNames:
    #             self.df.loc[self.df['index'].isin(Pins[groupName]),'color'] = EleGroupColors[groupName]
        
    #    # redraw: 
    #     self.fig.data[0]['marker']['color'] = self.df['color']
       



In [154]:
import pandas as pd
import numpy as np
import dash
import plotly.express as px
import plotly.graph_objects as go


sgdict = {
    'eleCath':[1,14,23,44,55],
    'eleAno':[2,15,24,56,78,19],
    'amplitude':150,
    'freq':130,
    'phase':50,
}

groupNum = 1
ncols = 4
eleCath = sgdict['eleCath']
eleAno  = sgdict['eleAno']

nrows = np.ceil(len(eleCath)/ncols) + np.ceil(len(eleAno)/ncols)
padding = lambda n: ncols*int(np.ceil(n/ncols)) - n
eleCathPadded = eleCath + [-1] * padding(len(eleCath))         # pad with -1's for spacing
eleAnoPadded  = eleAno  + [-1] * padding(len(eleAno ))         # pad with -1's

colorCath = ['red'] *  len(eleCathPadded)
colorAno =  ['blue'] * len(eleAnoPadded)

x, y = np.meshgrid(np.arange(0, 4), np.arange(0, nrows))
y = nrows - y #mirror

df = pd.DataFrame({'x': x.ravel(),
                   'y': y.ravel(),
                   'color':colorCath + colorAno,
                   'label':eleCathPadded + eleAnoPadded})

# pop the rows with -1 padding, everything will be properly aligned
df = df.loc[~df['label'].isin([-1])]

# plot the points
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x = df['x'].to_list(), 
        y = df['y'].to_list(), 
        text = df['label'].to_list(), 
        textfont=dict(
            family="sans serif",
            size=18,
            color="White"
        ),
        marker_color=df['color'],
        mode='markers+text',
        marker={'size': 40}
    ))


## if we wanted anode and cathode annotations:
# add annotations to the plot
# fig.add_annotation(
#             x=-1,
#             y=4,
#             text="<b>Cath:",
#             showarrow=False,
#             font=dict(
#                 size=24,
#             ),
#         )

# fig.add_annotation(
#             x=-1,
#             y=2,
#             text="<b>Ano:",
#             showarrow=False,
#             font=dict(
#                 size=24,
#             ),
#         )

# annotate amplitude, freq and phase for SG
# afp = (int(sgdict['amplitude']) , int(sgdict['freq']), int(sgdict['phase']))
# fig.add_annotation(
#             x=0,
#             y=0,
#             text="<b>Amp=%i, Freq=%i, Phase: %i" %afp,
#             showarrow=False,
#             font=dict(
#                 size=24,
#             ),
#         )



# make figure parametric based on contents:
pixelsPerRow = 50        # [pixels/unit]
sidePadding = 125        # [pixels]
topBottomPadding = 125   # [pixels]

leftLabelSpace = 0       # in graph units
bottomLabelSpace = 1

leftLabelPadding = pixelsPerRow * leftLabelSpace
bottomLabelPadding = pixelsPerRow * bottomLabelSpace

height = nrows*pixelsPerRow + bottomLabelPadding + 2*topBottomPadding
width  = ncols*pixelsPerRow + 2*sidePadding + leftLabelPadding

# update figure layout
fig.update_layout({
            'template': 'simple_white',
            'width':width,
            'height':height,
            'showlegend': False,
            'title':'Stim Group %i' %groupNum,
            'title_x':0.5,
            'title_y':0.85,
            'title_font_size':24
            #'margin':dict(l=20, r=20, b=20, pad=20)
            #'margin':dict(pad=0)
        })
fig.update_xaxes(mirror=True, showticklabels=False, ticks='')
fig.update_yaxes(mirror=True, showticklabels=False, ticks='')

#fig.update_xaxes(range=[-leftLabelSpace,ncols -.6])      #visual inspection
fig.update_yaxes(range=[-bottomLabelSpace,nrows + .75])  #visual inspection


# requirements / desires:
# a well balanced figure - regardless of the number of electrodes in the list
#   certian number of pixels per row?
#   a figure that resizes based on the number of electrodes, so the labels
#   all have the right amount of space. 
#   in each of the plots, all the dots are the same size,and realtive spacing. 
#   

In [130]:
df

,x,y,color,label
0,0,4.0,red,1
1,1,4.0,red,14
2,2,4.0,red,23
3,3,4.0,red,18
4,0,3.0,red,21
5,1,3.0,red,44
6,2,3.0,red,56
8,0,2.0,blue,2
9,1,2.0,blue,15
10,2,2.0,blue,24


### test out the figures performance in different senarios.


In [ ]:
# 3 of each
sgdict1 = 
{
    'eleCath':[1,14,23],
    'eleAno':[2,15,24],
    'amplitude':150
    'freq':130
    'phase':50
}

# 1 of each
sgdict2 = 
{
    'eleCath':[1],
    'eleAno':[2],
    'amplitude':150
    'freq':130
    'phase':50
}

# lots of electrodes
sgdict3 = 
{
    'eleCath':[1,14,23,18,21,44,56],
    'eleAno':[2,15,24,23,45,34],
    'amplitude':150
    'freq':130
    'phase':50
}

# only anodes
sgdict4 = 
{
    'eleCath':[1,14,23,18,21,44,56],
    'eleAno':[2,15,24,23,45,34],
    'amplitude':150
    'freq':130
    'phase':50
}

# only cathodes
sgdict5 = 
{
    'eleCath':[1,14,23,18,21,44,56],
    'eleAno':[2,15,24,23,45,34],
    'amplitude':150
    'freq':130
    'phase':50
}

# degenerate case - no electrodes?
sgdict6 = 
{
    'eleCath':[1,14,23,18,21,44,56],
    'eleAno':[2,15,24,23,45,34],
    'amplitude':150
    'freq':130
    'phase':50
}

sgdicts = [sgdict1,sgdict2,sgdict3,sgdict4,sgdict5,sgdict6]

# make a plot for each test case 
for sgd in sgdicts: 
    sg = StimGroup()
    sg.update(sgd)
    sg.fig.show()
    

